# Empirical stacked profiles split by detection significance q

We use the **fully empirical** aperture-normalised estimator (no pressure model, no central value $y_0$) and compare the stacked profile across bins of the catalogue detection significance `q_from_Y_5R500c`.

For each cluster $a$, in scaled radius $x=\theta/\theta_{500,a}$:
$$\bar y_{i,a}=\langle y_p\rangle_{i}\ (w_p=1),\qquad y_{{\rm norm},a}=\frac{Y_{500,a}}{\pi\theta_{500,a}^2},\quad Y_{500,a}=\!\!\sum_{\theta_p<\theta_{500,a}}\!\! y_p\,\Omega_{\rm pix},$$
$$g_{i,a}=\bar y_{i,a}/y_{{\rm norm},a},\qquad \hat f_i=\langle g_{i,a}\rangle\ (W_{i,a}=1).$$

Raw pixel values are used; **no background is subtracted**. We split into non-overlapping (differential) q bins so each stack is an independent sample. The Arnaud A10 projected GNFW reference is normalised the same aperture way and annulus-averaged in the same $x$ bins. The estimator is defined in this notebook; we do not use `flamingo.profiles.stacking`.

In [1]:
import os

# Limit JAX GPU memory allocation to 10% for prefilling
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.10'


import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt

# Publication-quality plot defaults
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "axes.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
})


from flamingo import paths
from flamingo.maps import read_map
from flamingo.catalogue import load_catalogue
from flamingo.geometry import query_disc_separation, ARCMIN_PER_RAD
from flamingo.profiles import projected_shape

path_highres = '/rds/rds-lxu/tsz_project/flamingo_highres_maps/y_unlensed_L2p8_m9_lc0_nside16384.fits'
catalogue_path = '/scratch/scratch-lxu/flamingo_repo/data/hydro_L2p8m9/catalogue/halo_catalogue_M500c_5e13_zlt3_y0q_arnaudB1.csv'

# ymap = read_map(paths.HYDRO_MAP)
ymap = read_map(path_highres)

# y0q catalogue has q/y0 but no sky positions; rows match HYDRO_CATALOGUE order.
df = load_catalogue(catalogue_path)
pos = pd.read_csv(
    paths.HYDRO_CATALOGUE,
    comment='#',
    usecols=['theta_rot_rad', 'phi_rot_rad'],
)
df[['theta_rot_rad', 'phi_rot_rad']] = pos
NSIDE = hp.npix2nside(ymap.size)
OMEGA_PIX = hp.nside2pixarea(NSIDE)   # solid angle per pixel [sr]
print('clusters', len(df), '| NSIDE', NSIDE)


clusters 1555542 | NSIDE 16384


In [2]:
# Non-overlapping detection-significance bins, capped for runtime.
# QCOL = 'q_from_Y_5R500c'
QCOL = 'q'
N_MAX = 4000          # max clusters per bin (random subsample if more)
RNG = 42
q = df[QCOL]

q_bins = [
    ('q > 30',      q > 30),
    ('10 < q < 30', (q > 10) & (q <= 30)),
    ('5 < q < 10',  (q > 5) & (q <= 10)),
    ('2 < q < 5',   (q > 2) & (q <= 5)),
]

samples = []
for label, mask in q_bins:
    sub = df[mask]
    n_all = len(sub)
    if n_all > N_MAX:
        sub = sub.sample(N_MAX, random_state=RNG)
    samples.append((label, sub))
    print(f'{label:12s}  N_total={n_all:5d}  N_used={len(sub):4d}  '
          f'median theta500={sub["theta500_arcmin"].median():.2f} arcmin')

q > 30        N_total=   26  N_used=  26  median theta500=15.72 arcmin
10 < q < 30   N_total=  510  N_used= 510  median theta500=8.59 arcmin
5 < q < 10    N_total= 1856  N_used=1856  median theta500=5.67 arcmin
2 < q < 5     N_total=11493  N_used=4000  median theta500=3.79 arcmin


In [3]:
# Scaled-radius bins x = theta / theta_500 (log-spaced), area-weighted centres.
x_edges = np.logspace(np.log10(0.01), np.log10(5.0), 101)   # 100 bins
x_lo, x_hi = x_edges[:-1], x_edges[1:]
x_mid = (2.0 / 3.0) * (x_hi**3 - x_lo**3) / (x_hi**2 - x_lo**2)


def empirical_profile(theta_c, phi_c, theta500_arcmin, x_edges):
    """Annular means (w_p=1) and aperture-mean normalisation y_norm. No y_0."""
    theta500_rad = theta500_arcmin / ARCMIN_PER_RAD
    pix, sep_rad = query_disc_separation(NSIDE, theta_c, phi_c, x_edges[-1] * theta500_rad)
    y_p = ymap[pix]
    x = sep_rad / theta500_rad

    ybar = np.full(len(x_edges) - 1, np.nan)
    idx = np.digitize(x, x_edges) - 1
    for b in range(ybar.size):
        sel = idx == b
        if sel.any():
            ybar[b] = y_p[sel].mean()

    inside = x < 1.0
    y_norm = (y_p[inside].sum() * OMEGA_PIX) / (np.pi * theta500_rad**2)
    return ybar, y_norm


def stack_empirical(sample, x_edges):
    """Stack g_{i,a} = ybar_{i,a} / y_norm,a over clusters (equal weights)."""
    g_rows = []
    for _, row in sample.iterrows():
        ybar, y_norm = empirical_profile(
            float(row['theta_rot_rad']), float(row['phi_rot_rad']),
            float(row['theta500_arcmin']), x_edges)
        if np.isfinite(y_norm) and y_norm > 0:
            g_rows.append(ybar / y_norm)
    G = np.vstack(g_rows)
    n = np.sum(np.isfinite(G), axis=0)
    fhat = np.nanmean(G, axis=0)
    sem = np.nanstd(G, axis=0) / np.sqrt(np.maximum(n, 1))
    return dict(fhat=fhat, sem=sem, n=G.shape[0])


def aperture_mean_inside_one(shape_func):
    """Area-weighted aperture mean of a projected model inside x < 1."""
    xa = np.linspace(0.0, 1.0, 800)
    return 2.0 * np.trapezoid(np.asarray(shape_func(xa)) * xa, xa)


def aperture_normalised_model(x_grid, shape_func):
    """Continuous aperture-normalised model, useful for visual shape checks."""
    return np.asarray(shape_func(x_grid)) / aperture_mean_inside_one(shape_func)


def binned_aperture_normalised_model(x_edges, shape_func, n_quad=200):
    """Model averaged over the same annuli as the stacked data.

    Data points are annular means, so the exact model comparison in bin i is
    2 int_lo^hi f_norm(x) x dx / (hi^2 - lo^2).
    """
    aperture_mean = aperture_mean_inside_one(shape_func)
    out = []
    for lo, hi in zip(x_edges[:-1], x_edges[1:]):
        xb = np.linspace(lo, hi, n_quad)
        fb = np.asarray(shape_func(xb)) / aperture_mean
        out.append(2.0 * np.trapezoid(fb * xb, xb) / (hi**2 - lo**2))
    return np.asarray(out)


stacks = [(label, stack_empirical(sub, x_edges)) for label, sub in samples]
for label, st in stacks:
    print(f'{label:12s}  stacked N={st["n"]}')

xx = np.logspace(np.log10(0.01), np.log10(5.0), 100)
arnaud = binned_aperture_normalised_model(x_edges, projected_shape)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


q > 30        stacked N=26
10 < q < 30   stacked N=510
5 < q < 10    stacked N=1856
2 < q < 5     stacked N=4000


## Comparison across q bins

All stacks share the aperture normalisation, so differences in shape are intrinsic, not amplitude offsets. Higher-q (more significant, more massive/resolved) clusters are expected to track the model more closely into the core; low-q clusters are noisier and approach the pixel/background floor at small $x$.

In [4]:
fig, ax = plt.subplots(figsize=(7.4, 5.6))
colors = plt.cm.viridis(np.linspace(0, 0.85, len(stacks)))
for c, (label, st) in zip(colors, stacks):
    ax.fill_between(x_mid, st['fhat'] - st['sem'], st['fhat'] + st['sem'],
                    color=c, alpha=0.25)
    ax.plot(x_mid, st['fhat'], 'o-', ms=4, lw=1.4, color=c,
            label=f'{label}  (N={st["n"]})')
ax.plot(x_mid, arnaud, 'k--', lw=1.8, label='Arnaud A10 (projected, aperture-norm, bin-averaged)')
ax.axvline(1.0, color='grey', ls=':', lw=1)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'$\theta / \theta_{500}$')
ax.set_ylabel(r'$\hat f = \langle \bar y / (Y_{500}/\pi\theta_{500}^2) \rangle$')
ax.set_ylim(1e-2, 3e1)
ax.set_title('Empirical stacked y-profile by detection significance q')
ax.legend(fontsize=9)
fig.tight_layout(); plt.show()

In [5]:
fig, ax = plt.subplots(figsize=(7.4, 5.6))
colors = plt.cm.viridis(np.linspace(0, 0.85, len(stacks)))
for c, (label, st) in zip(colors, stacks):
    ax.fill_between(x_mid, st['fhat'] - st['sem'], st['fhat'] + st['sem'],
                    color=c, alpha=0.25)
    ax.plot(x_mid, st['fhat'], 'o-', ms=4, lw=1.4, color=c,
            label=f'{label}  (N={st["n"]})')
ax.plot(x_mid, arnaud, 'k--', lw=1.8, label='Arnaud A10 (projected, aperture-norm, bin-averaged)')
ax.axvline(1.0, color='grey', ls=':', lw=1)
ax.set_xscale('linear'); ax.set_yscale('linear')
ax.set_xlabel(r'$\theta / \theta_{500}$')
ax.set_ylabel(r'$\hat f = \langle \bar y / (Y_{500}/\pi\theta_{500}^2) \rangle$')
ax.set_ylim(1e-2, 8)
ax.set_xlim(0, 3)
ax.set_title('Empirical stacked y-profile by detection significance q')
ax.legend(fontsize=9)
fig.tight_layout(); plt.show()

# Background-subtracted analysis

We now subtract a local background **first**, then run the same empirical aperture-normalised estimator. The background is the median $y$ in a $4$–$6\,\theta_{500}$ ring around each cluster, removed from every pixel before forming both the annular means $\bar y_{i,a}$ and the aperture normalisation $y_{{\rm norm},a}$. Removing the correlated/2-halo and sky floor should make the stacks agree with the projected Arnaud A10 shape more closely in the outskirts.

In [6]:
# Background ring in units of theta500. The local background b is the median y
# in this ring; it is removed from every pixel before forming both the annular
# means and the aperture normalisation, so g = (ybar_i - b)/(y_norm_raw - b).
BKG_IN, BKG_OUT = 3.0, 5.0


def empirical_profile_bkg(theta_c, phi_c, theta500_arcmin, x_edges,
                          bkg_in=BKG_IN, bkg_out=BKG_OUT):
    """Empirical estimator with a local background subtracted (median in ring)."""
    theta500_rad = theta500_arcmin / ARCMIN_PER_RAD
    r_out = max(x_edges[-1], bkg_out) * theta500_rad
    pix, sep_rad = query_disc_separation(NSIDE, theta_c, phi_c, r_out)
    y_p = ymap[pix]
    x = sep_rad / theta500_rad

    ring = (x >= bkg_in) & (x <= bkg_out)
    b = float(np.median(y_p[ring])) if ring.sum() > 5 else 0.0
    y_p = y_p - b

    ybar = np.full(len(x_edges) - 1, np.nan)
    idx = np.digitize(x, x_edges) - 1
    for k in range(ybar.size):
        sel = idx == k
        if sel.any():
            ybar[k] = y_p[sel].mean()

    inside = x < 1.0
    y_norm = (y_p[inside].sum() * OMEGA_PIX) / (np.pi * theta500_rad**2)
    return ybar, y_norm


def stack_empirical_bkg(sample, x_edges):
    """Stack the background-subtracted g_{i,a} over clusters (equal weights)."""
    g_rows = []
    for _, row in sample.iterrows():
        ybar, y_norm = empirical_profile_bkg(
            float(row['theta_rot_rad']), float(row['phi_rot_rad']),
            float(row['theta500_arcmin']), x_edges)
        if np.isfinite(y_norm) and y_norm > 0:
            g_rows.append(ybar / y_norm)
    G = np.vstack(g_rows)
    n = np.sum(np.isfinite(G), axis=0)
    fhat = np.nanmean(G, axis=0)
    sem = np.nanstd(G, axis=0) / np.sqrt(np.maximum(n, 1))
    return dict(fhat=fhat, sem=sem, n=G.shape[0])


stacks_bkg = [(label, stack_empirical_bkg(sub, x_edges)) for label, sub in samples]
for label, st in stacks_bkg:
    print(f'{label:12s}  stacked N={st["n"]}  (background-subtracted)')

q > 30        stacked N=26  (background-subtracted)
10 < q < 30   stacked N=510  (background-subtracted)
5 < q < 10    stacked N=1856  (background-subtracted)
2 < q < 5     stacked N=4000  (background-subtracted)


## Comparison: background-subtracted vs raw

Solid curves with markers are background-subtracted; the thin lines repeat the raw stacks from above. The subtraction mostly removes the outskirt floor ($x\gtrsim1$); the cores are unchanged. Bins that fall to zero or negative after subtraction (near and beyond the background ring) drop off the log axis, as expected.

In [7]:
fig, ax = plt.subplots(figsize=(7.4, 5.6))
colors = plt.cm.viridis(np.linspace(0, 0.85, len(stacks_bkg)))
for c, (label, st) in zip(colors, stacks_bkg):
    ax.fill_between(x_mid, st['fhat'] - st['sem'], st['fhat'] + st['sem'],
                    color=c, alpha=0.25)
    ax.plot(x_mid, st['fhat'], 'o-', ms=4, lw=1.4, color=c,
            label=f'{label}  (N={st["n"]})')
# raw (no background subtraction) as thin lines for comparison
for c, (label, st) in zip(colors, stacks):
    ax.plot(x_mid, st['fhat'], '-', lw=0.8, color=c, alpha=0.4)
ax.plot([], [], '-', lw=0.8, color='grey', alpha=0.6, label='raw (no bkg sub)')
ax.plot(x_mid, arnaud, 'k--', lw=1.8, label='Arnaud A10 (projected, aperture-norm, bin-averaged)')
ax.axvline(1.0, color='grey', ls=':', lw=1)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'$\theta / \theta_{500}$')
ax.set_ylabel(r'$\hat f = \langle \bar y / (Y_{500}/\pi\theta_{500}^2) \rangle$')
ax.set_ylim(1e-2, 3e1)
ax.set_title('Background-subtracted (solid) vs raw (thin) by q')
ax.legend(fontsize=9)
fig.tight_layout(); plt.show()

In [8]:
fig, ax = plt.subplots(figsize=(7.4, 5.6))
colors = plt.cm.viridis(np.linspace(0, 0.85, len(stacks_bkg)))
for c, (label, st) in zip(colors, stacks_bkg):
    ax.fill_between(x_mid, st['fhat'] - st['sem'], st['fhat'] + st['sem'],
                    color=c, alpha=0.25)
    ax.plot(x_mid, st['fhat'], 'o-', ms=4, lw=1.4, color=c,
            label=f'{label}  (N={st["n"]})')
# raw (no background subtraction) as thin lines for comparison
for c, (label, st) in zip(colors, stacks):
    ax.plot(x_mid, st['fhat'], '-', lw=0.8, color=c, alpha=0.4)
ax.plot([], [], '-', lw=0.8, color='grey', alpha=0.6, label='raw (no bkg sub)')
ax.plot(x_mid, arnaud, 'k--', lw=1.8, label='Arnaud A10 (projected, aperture-norm, bin-averaged)')
ax.axvline(1.0, color='grey', ls=':', lw=1)
ax.set_xscale('linear'); ax.set_yscale('linear')
ax.set_xlabel(r'$\theta / \theta_{500}$')
ax.set_ylabel(r'$\hat f = \langle \bar y / (Y_{500}/\pi\theta_{500}^2) \rangle$')
ax.set_ylim(1e-2, 8)
ax.set_xlim(0, 3)
ax.set_title('Background-subtracted (solid) vs raw (thin) by q')
ax.legend(fontsize=9)
fig.tight_layout(); plt.show()